<a href="https://colab.research.google.com/github/AditPradana36/semanticsegmentation_ADE20K/blob/main/semanticsegmentation_ADE20K.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📷 Semantic Segmentation of Panoramic Images (Google Colab Guide)

## Overview
This notebook performs **semantic segmentation** of panoramic street-level images using the [**MIT Semantic Segmentation PyTorch framework**](https://github.com/CSAILVision/semantic-segmentation-pytorch/tree/master?tab=readme-ov-file) pretrained on the **ADE20K dataset (150 classes)**.  
The workflow segments images, aggregates pixel counts by observation point, and exports both visual and tabular outputs for urban environment analysis.

---

## Environment
- Platform: **Google Colab**
- **GPU: Recommend for setting up runtime type to T4 GPU**
- Framework: PyTorch

Required libraries are installed automatically. Pretrained model weights are downloaded from the MIT CSAIL repository.

---


## 1. Setup: Download Semantic Segmentation Model (Run Once)

This cell prepares the Google Colab environment for semantic segmentation using the **MIT CSAIL semantic-segmentation-pytorch** framework.

### What this cell does:
1. Checks that the notebook is running in **Google Colab**
2. Installs required dependency (`yacs`)
3. Clones the MIT semantic segmentation repository
4. Downloads pretrained **ADE20K** model weights (ResNet50 + PPM)
5. Saves installation logs to `install.log` to keep output clean

### Important notes:
- This step is **required before running segmentation**
- The model is **pretrained** (no training required)
- The demo is **not executed** (`DOWNLOAD_ONLY=1`)
- You only need to run this cell **once per Colab session**

### When to re-run:
- If the Colab runtime is reset
- If you switch to a new Colab environment


In [ ]:
%%bash
# Colab-specific setup
!(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
pip install yacs 2>&1 >> install.log
git init 2>&1 >> install.log
git remote add origin https://github.com/CSAILVision/semantic-segmentation-pytorch.git 2>> install.log
git pull origin master 2>&1 >> install.log
DOWNLOAD_ONLY=1 ./demo_test.sh 2>> install.log

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
From https://github.com/CSAILVision/semantic-segmentation-pytorch
 * branch            master     -> FETCH_HEAD
 * [new branch]      master     -> origin/master


In [ ]:
# System libs
import os, csv, torch, numpy, scipy.io, PIL.Image, torchvision.transforms
# Our libs
from mit_semseg.models import ModelBuilder, SegmentationModule
from mit_semseg.utils import colorEncode

colors = scipy.io.loadmat('data/color150.mat')['colors']
names = {}
with open('data/object150_info.csv') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        names[int(row[0])] = row[5].split(";")[0]

def visualize_result(img, pred, index=None):
    # filter prediction class if requested
    if index is not None:
        pred = pred.copy()
        pred[pred != index] = -1
        print(f'{names[index+1]}:')

    # colorize prediction
    pred_color = colorEncode(pred, colors).astype(numpy.uint8)

    # aggregate images and save
    im_vis = numpy.concatenate((img, pred_color), axis=1)
    display(PIL.Image.fromarray(im_vis))

## 2. Load Libraries and Semantic Class Metadata

This cell initializes all required libraries and loads metadata needed for semantic segmentation and visualization.

### What this cell does:
- Imports system libraries for image processing, deep learning, and numerical computation
- Imports the MIT semantic segmentation model and utilities
- Loads the ADE20K color map for visualizing segmentation results
- Loads semantic class names (150 categories) from the ADE20K label file
- Defines a helper function to visualize segmentation results

### Output:
- `colors`: color palette for semantic classes
- `names`: dictionary mapping class IDs to human-readable labels
- `visualize_result()`: function to display original images alongside segmentation output

### Notes:
- This step is required before running the segmentation model
- Class labels follow the ADE20K dataset convention
- Visualization shows the original image and the color-coded segmentation side by side


In [ ]:
# Network Builders
net_encoder = ModelBuilder.build_encoder(
    arch='resnet50dilated',
    fc_dim=2048,
    weights='ckpt/ade20k-resnet50dilated-ppm_deepsup/encoder_epoch_20.pth')
net_decoder = ModelBuilder.build_decoder(
    arch='ppm_deepsup',
    fc_dim=2048,
    num_class=150,
    weights='ckpt/ade20k-resnet50dilated-ppm_deepsup/decoder_epoch_20.pth',
    use_softmax=True)

crit = torch.nn.NLLLoss(ignore_index=-1)
segmentation_module = SegmentationModule(net_encoder, net_decoder, crit)
segmentation_module.eval()
segmentation_module.cuda()

Loading weights for net_encoder
Loading weights for net_decoder


SegmentationModule(
  (encoder): ResnetDilated(
    (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): SynchronizedBatchNorm2d(64, eps=1e-05, momentum=0.001, affine=True, track_running_stats=True)
    (relu1): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): SynchronizedBatchNorm2d(64, eps=1e-05, momentum=0.001, affine=True, track_running_stats=True)
    (relu2): ReLU(inplace=True)
    (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn3): SynchronizedBatchNorm2d(128, eps=1e-05, momentum=0.001, affine=True, track_running_stats=True)
    (relu3): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(128, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): SynchronizedBatchNorm2d(64, eps=1

## 3. Image Loading and Normalization

This function loads an input image and prepares it for semantic segmentation.

### What this function does:
- Opens an image file and converts it to RGB format
- Converts the image to a PyTorch tensor
- Normalizes pixel values using ImageNet mean and standard deviation
- Returns both:
  - The normalized tensor (for model input)
  - The original image as a NumPy array (for visualization and saving)

### Notes:
- ImageNet normalization is required for compatibility with the pretrained ADE20K model
- The original image is preserved for side-by-side visualization


In [ ]:
# Function to load and normalize image
def load_and_normalize_image(image_path):
    pil_to_tensor = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225])
    ])
    pil_image = PIL.Image.open(image_path).convert('RGB')
    img_data = pil_to_tensor(pil_image)
    return img_data, np.array(pil_image)

## 4. Run Semantic Segmentation and Export Results

This section mounts Google Drive, runs semantic segmentation on all images in a directory, saves visualization outputs, and exports pixel statistics to a CSV file.

### What this section does:
1. **Mounts Google Drive**
   - Enables reading input images and saving outputs directly to Drive

2. **Defines output-saving function**
   - Combines the original image and its color-coded segmentation
   - Saves the result as a single image file

3. **Sets input and output directories**
   - `image_directory`: folder containing input images
   - `output_directory`: folder for segmented images
   - `output_path`: CSV file storing pixel counts

4. **Runs segmentation for each image**
   - Loads and normalizes the image
   - Runs the pretrained ADE20K segmentation model
   - Predicts a semantic label for each pixel

5. **Counts pixels per semantic class**
   - Computes pixel frequency for all detected classes
   - Maps class IDs to ADE20K class names

6. **Aggregates results by observation point**
   - Extracts `pointID` from filenames
   - Sums pixel counts across image slices belonging to the same point

7. **Exports results**
   - Saves segmented images to Google Drive
   - Writes aggregated pixel counts to a CSV file

---

### Outputs:
- **Segmented images**:  
  `/results/segmented_<original_filename>.jpg`

- **CSV file**:  
  `segmentation_results_sliced.csv`  
  Each row represents one observation point with pixel counts per semantic category.

---

### Notes:
- GPU acceleration is used automatically if available
- Filename consistency is required for correct `pointID` aggregation
- Pixel counts can be used for further quantitative or spatial analysis


In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title
# Function to save segmented image
def save_segmented_image(img, pred, output_path):
    pred_color = colorEncode(pred, colors).astype(np.uint8)
    im_vis = np.concatenate((img, pred_color), axis=1)
    segmented_image = PIL.Image.fromarray(im_vis)
    segmented_image.save(output_path)

In [ ]:
# @title
import os
import numpy as np  # Add this line to import NumPy
import PIL.Image
import pandas as pd  # Add this line to import pandas
import torch

In [ ]:
# Set up directories
image_directory = "/content/drive/MyDrive/Dummy/try_05012025"  #@param {type:"string"}
output_directory = "/content/drive/MyDrive/Dummy/try_05012025/results"  #@param {type:"string"}
output_path = "/content/drive/MyDrive/Dummy/try_05012025/results/segmentation_results_sliced.csv"  #@param {type:"string"}


# Ensure the output directory exists
os.makedirs(output_directory, exist_ok=True)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_results = {}
# Get a list of all image files in the directory
image_files = [f for f in os.listdir(image_directory) if os.path.isfile(os.path.join(image_directory, f))]
# Process each image
for image_file in image_files:
    image_path = os.path.join(image_directory, image_file)

    # Load and normalize image
    img_data, img_original = load_and_normalize_image(image_path)

    # Convert numpy array to tensor
    img_data = torch.tensor(img_data).to(device)

    # Ensure proper batch dimension and move to GPU if available
    singleton_batch = {'img_data': img_data.unsqueeze(0)}
    output_size = img_data.shape[1:]

    # Run the segmentation module
    with torch.no_grad():
        scores = segmentation_module(singleton_batch, segSize=output_size)

    # Get the predicted scores for each pixel
    _, pred = torch.max(scores, dim=1)
    pred = pred.cpu().numpy()[0]  # Convert back to numpy on CPU for processing

    # Calculate the pixel count of each segmented object
    unique, counts = np.unique(pred, return_counts=True)

    # Assuming 'names' is a dictionary mapping class indices to labels
    pixel_counts = {names[k + 1]: v for k, v in zip(unique, counts)}

    # Extract the pointID from the image name (e.g., 'Salinan pointID_X_heading_X_pitch_0')
    pointID = '_'.join(image_file.split('_')[1:3])  # Extract "pointID_X"

    # Initialize or update the result for the current pointID
    if pointID not in all_results:
        all_results[pointID] = {'Image': pointID, 'total_pixels': 0, 'counts': {}}

    # Update counts for the current slice
    all_results[pointID]['total_pixels'] += pred.size
    for k, v in zip(unique, counts):
        label = names[k + 1]
        if label in all_results[pointID]['counts']:
            all_results[pointID]['counts'][label] += v
        else:
            all_results[pointID]['counts'][label] = v

    # Save segmented image
    segmented_image_path = os.path.join(output_directory, f'segmented_{image_file}')
    save_segmented_image(img_original, pred, segmented_image_path)

# Prepare final results with pixel counts
final_results = []
for result in all_results.values():
    pixel_counts = {label: count for label, count in result['counts'].items()}
    final_result = {'Image': result['Image']}
    final_result.update(pixel_counts)
    final_results.append(final_result)

# Convert to DataFrame and save to CSV
df = pd.DataFrame(final_results)
df.to_csv(output_path, index=False)
print(f"Segmentation results with pixel counts saved to '{output_path}'")


/tmp/ipython-input-3661945141.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  img_data = torch.tensor(img_data).to(device)


Segmentation results with pixel counts saved to '/content/drive/MyDrive/Dummy/try_05012025/results/segmentation_results_sliced.csv'


## (OPTIONAL) 5. Extract Segmentation-Only Images

This section extracts **only the semantic segmentation output** from combined images where the format is:

### What this code does:
- Uses the segmentation output directory as the input source
- Creates a subfolder (`just_segmented_image`) for cleaned segmentation images
- Crops each image from the **center to the right**, assuming the right half contains the segmentation
- Saves the cropped images with the original filenames

### Input:
- `output_directory`  
  Folder containing combined images (original + segmentation)

### Output:
- `output_directory/just_segmented_image/`  
  Folder containing segmentation-only images

### Notes:
- Assumes images are split evenly into two halves
- Supports `.jpg`, `.jpeg`, and `.png` formats
- Original images are not modified

This step is useful for preparing segmentation maps for visualization, further analysis, or dataset reuse.


In [ ]:
# @title
from PIL import Image
# Directory containing combined images (original | segmentation)
input_image_folder = output_directory

# Directory to save cropped segmentation-only images
output_image_folder = os.path.join(output_directory, 'just_segmented_image')


# Create output folder if it does not exist
os.makedirs(output_image_folder, exist_ok=True)

# Function to crop the right half of the image (segmentation result)
def crop_middle_to_right(image_path, output_path):
    with Image.open(image_path) as img:
        width, height = img.size

        # Crop from the middle to the right side
        left = width // 2
        top = 0
        right = width
        bottom = height

        cropped_img = img.crop((left, top, right, bottom))
        cropped_img.save(output_path)

# Process all images in the input folder
for file_name in os.listdir(input_image_folder):
    if file_name.lower().endswith(('png', 'jpg', 'jpeg')):
        input_path = os.path.join(input_image_folder, file_name)
        output_path = os.path.join(output_image_folder, file_name)

        crop_middle_to_right(input_path, output_path)
        print(f'Image "{file_name}" cropped successfully and saved to {output_path}')

print('All images have been processed successfully.')

Image "segmented_pointID_2036.jpg" cropped successfully and saved to /content/drive/MyDrive/Dummy/try_05012025/results/just_segmented_image/segmented_pointID_2036.jpg
Image "segmented_pointID_2016.jpg" cropped successfully and saved to /content/drive/MyDrive/Dummy/try_05012025/results/just_segmented_image/segmented_pointID_2016.jpg
Image "segmented_pointID_2027.jpg" cropped successfully and saved to /content/drive/MyDrive/Dummy/try_05012025/results/just_segmented_image/segmented_pointID_2027.jpg
Image "segmented_pointID_2035.jpg" cropped successfully and saved to /content/drive/MyDrive/Dummy/try_05012025/results/just_segmented_image/segmented_pointID_2035.jpg
Image "segmented_pointID_2029.jpg" cropped successfully and saved to /content/drive/MyDrive/Dummy/try_05012025/results/just_segmented_image/segmented_pointID_2029.jpg
Image "segmented_pointID_2022.jpg" cropped successfully and saved to /content/drive/MyDrive/Dummy/try_05012025/results/just_segmented_image/segmented_pointID_2022.jp

## 🧩 Optional Post-Processing: Spatial Entropy of Vegetation Distribution

This optional module computes **spatial entropy** to quantify the **spatial heterogeneity of vegetation distribution** within each image, complementing pixel-based vegetation metrics (e.g., GVI).  
The module is **fully independent** and does **not alter or interfere with** the core semantic segmentation pipeline.

### Method Overview
- A **binary vegetation mask** is derived from semantic segmentation outputs by aggregating vegetation-related classes.
- **Overlapping square sliding windows** are applied to the vegetation mask.
- For each window, **Shannon entropy** is computed from the proportion of vegetation pixels within the window.
- The final spatial entropy score is the **mean entropy across all windows**, representing overall spatial heterogeneity.

### Window Definition
- **Window size** is defined as **45% of the image’s smaller dimension**, ensuring scale invariance across images with heterogeneous resolutions.
- **Stride** is set to **5% of the window size**, producing overlapping windows and stable entropy estimates.
- This configuration follows a sensitivity analysis identifying window scales that best capture spatial heterogeneity.

### Interpretation
- Spatial entropy values are **normalized and bounded in [0, 1]**.
- **Higher values** indicate more spatially mixed, fragmented, or heterogeneous vegetation patterns.
- **Lower values** indicate more spatially uniform or clustered vegetation distributions.

### Integration Notes
- Run this module **after semantic segmentation**.
- Results are saved to a **separate CSV file** (e.g., `spatial_entropy_only.csv`).
- Outputs can be **merged post hoc** with existing metrics using the image identifier.

### Methodological Notes
- Absolute entropy values may differ depending on vegetation mask derivation; however, the metric robustly captures **relative spatial heterogeneity**.
- Spatial entropy is intended as a **comparative indicator**, suitable for cross-image and cross-location analyses.

This modular design preserves reproducibility, avoids interference with core outputs, and enables optional inclusion of spatial configuration metrics in downstream analyses.


In [ ]:
import numpy as np
import cv2

In [ ]:
def shannon_entropy_binary(p, eps=1e-7):
    if p <= eps or p >= 1.0 - eps:
        return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

In [ ]:
def spatial_entropy_equivalent(
    vegetation_mask,
    window_fraction=0.45
):
    """
    Mechanically equivalent to spatial_entropy_2d_relative_optimized,
    except mask derivation.
    """

    # Ensure binary float mask
    mask = (vegetation_mask > 0).astype(np.float32)

    H, W = mask.shape
    min_dim = min(H, W)

    # Window size (relative, min 4, odd)
    window_size = max(4, int(window_fraction * min_dim))
    if window_size % 2 == 0:
        window_size += 1

    # Dense sliding window via convolution
    kernel = np.ones((window_size, window_size), dtype=np.float32)

    # Zero-padding at borders (same as cv2.BORDER_CONSTANT)
    sum_windows = cv2.filter2D(mask, -1, kernel, borderType=cv2.BORDER_CONSTANT)

    # Match valid window region
    sum_windows = sum_windows[window_size - 1 : H, window_size - 1 : W]

    window_area = window_size * window_size
    p = sum_windows / window_area

    # Compute entropy map
    entropy_map = np.zeros_like(p, dtype=np.float32)

    valid = (p > 0) & (p < 1)
    entropy_map[valid] = (
        -p[valid] * np.log2(p[valid])
        - (1 - p[valid]) * np.log2(1 - p[valid])
    )

    return float(np.mean(entropy_map))


In [ ]:
VEGETATION_CLASSES = {
    "tree",
    "vegetation",
    "grass",
    "plant",
    "bush"
}

entropy_results = []

In [ ]:
for image_file in image_files:
    image_path = os.path.join(image_directory, image_file)

    img_data, _ = load_and_normalize_image(image_path)
    img_data = img_data.to(device)

    with torch.no_grad():
        scores = segmentation_module(
            {'img_data': img_data.unsqueeze(0)},
            segSize=img_data.shape[1:]
        )

    _, pred = torch.max(scores, dim=1)
    pred = pred.cpu().numpy()[0]

    vegetation_mask = np.zeros_like(pred, dtype=np.uint8)

    for class_idx, class_name in names.items():
        if class_name.lower() in VEGETATION_CLASSES:
            vegetation_mask[pred == (class_idx - 1)] = 1

    entropy_value = spatial_entropy_equivalent(
        vegetation_mask,
        window_fraction=0.45
    )

    pointID = '_'.join(image_file.split('_')[1:3])

    entropy_results.append({
        'Image': pointID,
        'spatial_entropy': entropy_value
    })


In [ ]:
entropy_df = pd.DataFrame(entropy_results)

entropy_output_path = (
    "/content/drive/MyDrive/Dummy/try_05012025/results/"
    "spatial_entropy_only_.csv"
)

entropy_df.to_csv(entropy_output_path, index=False)

print(f"Spatial entropy saved to '{entropy_output_path}'")


Spatial entropy saved to '/content/drive/MyDrive/Dummy/try_05012025/results/spatial_entropy_only_.csv'
